# Q1. 학생 성적 보고서 생성기

### (1) 코드

In [3]:
import csv
import json
import logging

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    results = []
    success_count = 0

    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                name = row["이름"]
                sid = row["학번"]

                try:
                    # 빈칸 확인
                    if not row["중간"] or not row["기말"] or not row["과제"]:
                        raise ValueError("결측값 존재")

                    mid = int(row["중간"])
                    fin = int(row["기말"])
                    hw = int(row["과제"])

                    # 중간 30%, 기말 50%, 과제 20%
                    avg = mid * 0.3 + fin * 0.5 + hw * 0.2

                    # 등급
                    if avg >= 90: grade = "A"
                    elif avg >= 80: grade = "B"
                    elif avg >= 70: grade = "C"
                    else: grade = "F"

                except ValueError:
                    # 결측값 있는 경우 None으로 처리
                    mid = int(row["중간"]) if row["중간"] else None
                    fin = int(row["기말"]) if row["기말"] else None
                    hw = int(row["과제"]) if row["과제"] else None
                    avg = None
                    grade = None

                student_data = {
                    "이름": name,
                    "학번": sid,
                    "점수": {"중간": mid, "기말": fin, "과제": hw},
                    "평균": avg,
                    "등급": grade
                }
                results.append(student_data)
                success_count += 1

                logging.info(f"{name}: {avg if avg is not None else '결측'}, {grade if grade is not None else '결측'}")

        # JSON 저장
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        return success_count

    except FileNotFoundError:
        logging.warning(f"파일이 없습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류: {csv_path}")
        return 0

### (2) 실행 결과

In [ ]:
[INFO] 김언어: 89.5, B
[INFO] 이국문: 84.4, B
[INFO] 박영문: 93.5, A
[INFO] 최역사: 결측, 결측

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None



처리된 학생 수: 4

[report.json 내용]
[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {
            "중간": 85,
            "기말": 92,
            "과제": 90
        },
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {
            "중간": 78,
            "기말": 88,
            "과제": 85
        },
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {
            "중간": 95,
            "기말": 90,
            "과제": 100
        },
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {
            "중간": null,
            "기말": 82,
            "과제": 88
        },
        "평균": null,
        "등급": null
    }
]

[FileNotFoundError 테스트]


0

### (3) 설명

*   **결측값 처리 전략**: csv.DictReader로 읽어온 각 행에서 빈 값이 발견되면 해당 학생의 평균과 등급은 None 으로 할당하여 JSON 저장 시 null로 나타나도록 처리했습니다.
*   **인코딩 선택의 근거**: 한글 데이터가 포함된 csv이므로 표준인 utf-8를 명시하였습니다.  JSON 저장 시 ensure_ascii=False 옵션을 사용하여 한글이 깨져보이지 않도록 했습니다.




---



# Q2. 사용자 정의 예외와 자모 분류

### (1) 코드 (2) 실행 결과

In [2]:
class InvalidJamoError(ValueError):
    """한글 자모 범위가 아닌 문자가 들어왔을 때 발생하는 예외"""
    pass

def classify_jamo(c: str) -> str:
    # 타입 검증
    if not isinstance(c, str):
        raise TypeError(f"문자열이어야 합니다 (입력: {type(c)})")

    # 길이 검증
    if len(c) != 1:
        raise ValueError(f"한 글자만 입력 가능합니다 (입력 길이: {len(c)})")

    code = ord(c)

    # 자음 범위: U+3131 ~ U+314E
    if 0x3131 <= code <= 0x314E:
        return "자음"
    # 모음 범위: U+314F ~ U+3163
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c}")

# 테스트 실행
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        # InvalidJamoError가 ValueError의 자식이므로 먼저 처리해야 함
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: 가
[ValueError] 한 글자만 입력 가능합니다 (입력 길이: 2)
[TypeError] 문자열이어야 합니다 (입력: <class 'int'>)
자음
모음
[ValueError] 한 글자만 입력 가능합니다 (입력 길이: 0)


### (3) 설명

InvalidJamoError는 입력된 값의 '내용'이 유효하지 않은 경우에 해당하므로, '부적절한 값'을 의미하는 ValueError의 자식으로 만드는 것이 논리적으로 적절합니다
. 예외 처리 시 자식 클래스인 InvalidJamoError를 부모인 ValueError보다 먼저 배치하여 구체적인 오류 메시지가 출력되도록 구성했습니다
